# DistilBERT Notebook (Train + Evaluate + Rule-based Chat Test)

### Install + Imports

In [1]:
# If needed (Colab / fresh environment), uncomment:
# !pip -q install transformers datasets accelerate evaluate scikit-learn pandas numpy torch

import os
import re
import json
import random
import numpy as np
import pandas as pd
import transformers

import torch
from sklearn.metrics import classification_report, confusion_matrix

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)

set_seed(42)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


e:\Conestoga\Level2\FRI-MORN-Project in ML\FinalProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch: 2.10.0+cpu
CUDA available: False


### Paths to V2/V2.6 datasets

In [2]:
EMOTION_CSV = "./data/merged/nurturejoy_v26_emotion_with_pregnancy_multisent_mixed.csv"
SAFETY_CSV  = "./data/merged/nurturejoy_v2_safety_merged_balanced.csv"

assert os.path.exists(EMOTION_CSV), f"Missing {EMOTION_CSV}"
assert os.path.exists(SAFETY_CSV), f"Missing {SAFETY_CSV}"

print("Found datasets ✅")


Found datasets ✅


### Basic cleaning

In [3]:
def clean_text(t: str) -> str:
    t = str(t).strip()
    t = re.sub(r"\s+", " ", t)
    return t


## PART A — DistilBERT Emotion Model (6-class)

### Load Emotion dataset + encode labels

In [4]:
emo_df = pd.read_csv(EMOTION_CSV).dropna(subset=["text", "category"]).copy()
emo_df["text"] = emo_df["text"].map(clean_text)
emo_df = emo_df[emo_df["text"].str.len() >= 5]

label_list_emo = ["POSITIVE", "NEUTRAL", "ANXIETY", "STRESS", "LOW_MOOD", "HIGH_DISTRESS"]
label2id_emo = {l:i for i,l in enumerate(label_list_emo)}
id2label_emo = {i:l for l,i in label2id_emo.items()}

emo_df = emo_df[emo_df["category"].isin(label_list_emo)].copy()
emo_df["label"] = emo_df["category"].map(label2id_emo)

print("Emotion distribution:")
print(emo_df["category"].value_counts())
emo_df.head(3)


Emotion distribution:
category
STRESS           16790
LOW_MOOD         15301
POSITIVE         15298
NEUTRAL          15293
HIGH_DISTRESS    15281
ANXIETY           6365
Name: count, dtype: int64


,text,category,source,label
0,oh my gosh,ANXIETY,maternal_combined,2
1,"trouble sleeping, confused mind, restless hear...",ANXIETY,maternal_combined,2
2,"All wrong, back off dear, forward doubt. Stay ...",ANXIETY,maternal_combined,2


### Train/Val/Test split

In [5]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    emo_df, test_size=0.30, random_state=42, stratify=emo_df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df["label"]
)

print("Emotion split sizes:", len(train_df), len(val_df), len(test_df))


Emotion split sizes: 59029 12649 12650


### Create HuggingFace Datasets

In [6]:
train_ds = Dataset.from_pandas(train_df[["text", "label"]].reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df[["text", "label"]].reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df[["text", "label"]].reset_index(drop=True))

train_ds, val_ds, test_ds


(Dataset({
     features: ['text', 'label'],
     num_rows: 59029
 }),
 Dataset({
     features: ['text', 'label'],
     num_rows: 12649
 }),
 Dataset({
     features: ['text', 'label'],
     num_rows: 12650
 }))

### Tokenizer + Tokenization

In [7]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True)

train_tok = train_ds.map(tokenize_batch, batched=True)
val_tok   = val_ds.map(tokenize_batch, batched=True)
test_tok  = test_ds.map(tokenize_batch, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


Map: 100%|██████████| 12650/12650 [00:01<00:00, 9935.28 examples/s] 


### Build Emotion Model

In [8]:
emotion_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list_emo),
    id2label=id2label_emo,
    label2id=label2id_emo
)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 574.51it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Metrics

In [9]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    # You can add F1 later; keep simple for now
    acc = (preds == labels).mean()
    return {"accuracy": acc}


### Train Emotion DistilBERT

In [19]:
emotion_args = TrainingArguments(
    output_dir="./distilbert_emotion_v2",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

emotion_trainer = Trainer(
    model=emotion_model,
    args=emotion_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

emotion_trainer.train()


e:\Conestoga\Level2\FRI-MORN-Project in ML\FinalProject\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 